# Q1. 233. Attention score





Compute how much each token attends to every other token using scaled dot-product attention.

$$A = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)$$

- $Q, K$ shape: `(seq_len, d_k)` → scores shape: `(seq_len, seq_len)`
- $A[i][j]$ = how much token $i$ attends to token $j$
- Softmax is applied **row-wise** (each row sums to 1)

**Input**
```
Q = [[1.0, 0.0],
     [0.0, 1.0]]

K = [[1.0, 0.0],
     [0.0, 1.0]]
```
**Expected Output**
```
[[0.6225, 0.3775],
 [0.3775, 0.6225]]

d_k = 2,  scale = √2 ≈ 1.414

S = Q @ Kᵀ / √2 = [[1, 0], [0, 1]] / 1.414 = [[0.707, 0.0], [0.0, 0.707]]

row 0: softmax([0.707, 0.0]) → exp([0.707,0]) = [2.028, 1.0] → [0.6225, 0.3775]
row 1: softmax([0.0, 0.707]) → exp([0,0.707]) = [1.0, 2.028] → [0.3775, 0.6225]
```

In [5]:
import numpy as np


def attention_scores(Q, K):
    """
    Computes scaled dot-product attention scores (softmax over keys) for
    one head.

    Args:
        Q (np.ndarray): Query matrix with shape (seq_len, d_k).
        K (np.ndarray): Key matrix with shape (seq_len, d_k).

    Returns:
        np.ndarray: Attention matrix A with shape (seq_len, seq_len),
        where A[i][j] is how much token i attends to token j.
    """
    # Scale by sqrt(d_k)
    d_k = Q.shape[1]
    scores = (Q @ K.T) / np.sqrt(d_k)
    print(scores)

    # Numerically stable softmax (row-wise)
    max_scores = np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores - max_scores)
    softmax_scores = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

    return softmax_scores
Q = np.array( [[1.0, 0.0],
     [0.0, 1.0]])

K = np.array([[1.0, 0.0],
     [0.0, 1.0]])

result = attention_scores(Q, K)
result

[[0.70710678 0.        ]
 [0.         0.70710678]]


array([[0.66976155, 0.33023845],
       [0.33023845, 0.66976155]])

# Q2. 811. Attention-weighted value sum




- same as above question - just '@V' at the end

Full single-head attention — compute attention weights then use them to mix value vectors.

$$S = \frac{QK^T}{\sqrt{d_k}}, \qquad A = \text{softmax}(S), \qquad O = AV$$

- $S$ shape: `(n, m)` — score of each query against each key
- $A$ shape: `(n, m)` — attention weights, rows sum to 1
- $O$ shape: `(n, d_v)` — weighted mix of value vectors

**Input**
```
Q = [[1.0, 0.0]]          shape (1, 2)
K = [[1.0, 0.0],
     [0.0, 1.0]]          shape (2, 2)
V = [[10.0,  0.0],
     [ 0.0, 20.0]]        shape (2, 2)
```
**Expected Output**
```
array([[6.69761549, 6.60476901]])

S = Q @ Kᵀ / √2 
A = softmax(S)  
O = A @ V
```

In [11]:
import numpy as np


def attention_weighted_value_sum(Q, K, V):
    """
    Computes the attention-weighted value sum (single-head attention output).

    Args:
        Q (np.ndarray): Queries of shape (n, d_k).
        K (np.ndarray): Keys of shape (m, d_k).
        V (np.ndarray): Values of shape (m, d_v).

    Returns:
        np.ndarray: Output O of shape (n, d_v).
    """
    # Convert inputs to NumPy arrays for efficient computation
    q_mat = np.asarray(Q, dtype=float)
    k_mat = np.asarray(K, dtype=float)
    v_mat = np.asarray(V, dtype=float)

    # Compute scaled dot-product scores
    d_k = q_mat.shape[1]
    scale = np.sqrt(d_k)
    scores = (q_mat @ k_mat.T) / scale


    # Apply row-wise, numerically stable softmax
    row_max = np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores - row_max)
    attn = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    print(attn)

    # Compute the attention-weighted sum of values
    return attn @ v_mat

Q = [[1.0, 0.0]]        
K = [[1.0, 0.0],
     [0.0, 1.0]]          
V = [[10.0,  0.0],
     [ 0.0, 20.0]]       

result = attention_weighted_value_sum(Q, K, V)
result

[[0.66976155 0.33023845]]


array([[6.69761549, 6.60476901]])

# Q3. 1082. Masked attention scores




Block certain key positions before softmax so they receive zero attention weight.

$$S = \frac{QK^T}{\sqrt{d}}, \qquad A = \text{softmax}(S + M), \quad M_{ij} = \begin{cases} 0 & \text{allow} \\ -10^9 & \text{mask} \end{cases}$$

Adding $-10^9$ makes $e^{-10^9} \approx 0$ after softmax — effectively zero probability.

**Input**
```
Q    = [[1.0, 0.0],
        [0.0, 1.0]]

K    = [[1.0, 0.0],
        [0.0, 1.0]]

mask = [[1, 0],   # query 0 → only key 0 allowed
        [1, 1]]   # query 1 → both keys allowed
```
**Expected Output**
```
[[1.0,    0.0   ],
 [0.3775, 0.6225]]

S = Q @ Kᵀ / √d → after mask applied:
  row 0: [s00,  -1e9]  → softmax → [1.0,    0.0   ]  ← key 1 zeroed out
  row 1: [s10,   s11]  → softmax → [0.3775, 0.6225]  ← both keys contribute
```

In [ ]:
import numpy as np


def masked_attention_scores(Q, K, mask):
    """
    Computes masked, row-wise attention probabilities for scaled dot-product
    attention.

    Args:
        Q (np.ndarray): Query matrix of shape (n, d).
        K (np.ndarray): Key matrix of shape (m, d).
        mask (np.ndarray): Mask of shape (n, m), where 1 means
            "allow" and 0 means "mask out".

    Returns:
        np.ndarray: Attention probability matrix A of shape (n, m).
    """
    # Compute scaled dot-product scores
    d = Q.shape[1]
    scale = np.sqrt(d)
    scores = (Q @ K.T) / scale

    # Apply mask: add large negative value where mask is 0
    scores = scores + (mask - 1.0) * 1e9

    # Numerically stable softmax per row
    max_scores = np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores - max_scores)
    sum_exp = np.sum(exp_scores, axis=1, keepdims=True)
    probs = exp_scores / sum_exp

    return probs


Q    = np.array([[1.0, 0.0],
        [0.0, 1.0]])

K    = np.array([[1.0, 0.0],
        [0.0, 1.0]])

mask = np.array([[1, 0],   # query 0 → only key 0 allowed
                [1, 1]] )  # query 1 → both keys allowed


result = masked_attention_scores(Q, K, mask)
result

array([[1.        , 0.        ],
       [0.33023845, 0.66976155]])

In [23]:
# generate a mask
np.tril(np.ones((4,4))) 

array([[1., 0., 0., 0.],
       [1., 1., 0., 0.],
       [1., 1., 1., 0.],
       [1., 1., 1., 1.]])

In [ ]:
vocab = ["I", "love", "you", "too", "this", "is", "a", "toy", "LLM", "<eos>"]
vocab_to_id = {w: i for i, w in enumerate(vocab)}
id_to_vocab = {i: w for w, i in vocab_to_id.items()}
np.random.seed(42)

V = len(vocab)   # 10
d = 3            # embedding dim

# embedding
E = np.random.randn(V, d)

W_Q = np.random.randn(d, d)
W_K = np.random.randn(d, d)
W_V = np.random.randn(d, d)

# 4️⃣ Output (prediction) layer
W_O = np.random.randn(d, V)

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()


def causal_attention(Q, K, V):
    T = Q.shape[0]
    scores = Q @ K.T / np.sqrt(d)
    print("attenstion before masking")
    print(scores)

    # causal mask
    mask = np.triu(np.ones((T, T)), k=1) * -1e9
    print(mask)
    scores = scores + mask

    print("attenstion after masking")
    print(scores)
    # softmaxs
    weights = np.exp(scores - np.max(scores, axis=1, keepdims=True))
    weights /= weights.sum(axis=1, keepdims=True)

    print("after softmax")
    print(weights)

    return weights @ V


def prefill(prompt_tokens):
    print("\n=== PREFILL ===")

    # embeddings
    X = np.stack([E[vocab_to_id[t]] for t in prompt_tokens])
    print("Input embeddings X:")
    print(X)

    # Q,K,V
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V

    print(Q)
    print(K)
    print(V)

    # output
    O = causal_attention(Q, K, V)

    print("\nFinal contextual representations O:")
    print(O)

    # return last token representation + KV cache
    return O[-1], K, V

def predict_next_token(h):
    logits = h @ W_O
    probs = softmax(logits)

    token_id = np.argmax(probs)
    return id_to_vocab[token_id], probs

prompt = ["I", "love"]

h_last, K_cache, V_cache = prefill(prompt)

next_token, probs = predict_next_token(h_last)

print("\nPredicted next token:")
print(next_token)

print("\nProbability distribution:")
for i, p in enumerate(probs):
    print(f"{id_to_vocab[i]:>4}: {p:.3f}")
